In [1]:
import os
import json
import glob
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
BASEDIR = '/Users/egyptdj/github/bruit-classification'
DATADIR = os.path.join(BASEDIR, 'data_v5')
MODELDIR = os.path.join(BASEDIR, 'twostep_model')

MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
SAMPLE_RATE = 16000
BATCH_SIZE = 1
SEED = 0

In [4]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [5]:
train_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'train', '**/*.wav')), dtype='object')
train_audio_label_list = [f.split('/')[-2] for f in train_audio_file_list]

In [6]:
val_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'val', '**/*.wav')), dtype='object')
val_audio_label_list = [f.split('/')[-2] for f in val_audio_file_list]

In [7]:
test_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'test', '**/*.wav')), dtype='object')
test_audio_label_list = [f.split('/')[-2] for f in test_audio_file_list]

In [8]:
class_name_dict = {}
i = 0
for label in train_audio_label_list:
  if not label in list(class_name_dict.keys()):
    class_name_dict[label] = i
    i += 1

In [9]:
with open(os.path.join(BASEDIR, 'class_name_dict.json'), 'w') as f:
  json.dump(class_name_dict, f)

In [10]:
train_audio_label_list = pd.Series([class_name_dict[i] for i in train_audio_label_list], dtype='int64')
val_audio_label_list = pd.Series([class_name_dict[i] for i in val_audio_label_list], dtype='int64')
test_audio_label_list = pd.Series([class_name_dict[i] for i in test_audio_label_list], dtype='int64')

In [11]:
train_audio_label_list_step1 = train_audio_label_list.map(lambda x: 0 if x==class_name_dict['background'] else 1)
val_audio_label_list_step1 = val_audio_label_list.map(lambda x: 0 if x==class_name_dict['background'] else 1)
test_audio_label_list_step1 = test_audio_label_list.map(lambda x: 0 if x==class_name_dict['background'] else 1)

In [12]:
train_audio_label_list_step2 = train_audio_label_list.replace({class_name_dict['background']: -1, class_name_dict['normal']: 0, class_name_dict['bruit']: 1, class_name_dict['normal-noise']: 2, class_name_dict['bruit-noise']: 2})
val_audio_label_list_step2 = val_audio_label_list.replace({class_name_dict['background']: -1, class_name_dict['normal']: 0, class_name_dict['bruit']: 1, class_name_dict['normal-noise']: 2, class_name_dict['bruit-noise']: 2})
test_audio_label_list_step2 = test_audio_label_list.replace({class_name_dict['background']: -1, class_name_dict['normal']: 0, class_name_dict['bruit']: 1, class_name_dict['normal-noise']: 2, class_name_dict['bruit-noise']: 2})

In [13]:
train_ds1 = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list_step1))
val_ds1 = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list_step1))
test_ds1 = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list_step1))

train_ds2 = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list_step2)).filter(lambda x, y: tf.math.logical_not(tf.math.equal(y, -1)))
val_ds2 = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list_step2)).filter(lambda x, y: tf.math.logical_not(tf.math.equal(y, -1)))
test_ds2 = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list_step2)).filter(lambda x, y: tf.math.logical_not(tf.math.equal(y, -1)))

In [14]:
# *.wav 파일을 경로에서부터 오디오파일로 읽어오기

@tf.function
def load_wav_and_label(audio_file, label):
  audio = tf.io.read_file(audio_file)
  audio, sample_rate = tf.audio.decode_wav(audio)
  audio = tf.squeeze(audio, axis=-1)
  return audio, label

In [15]:
yamnet_model = hub.load(MODEL_HANDLE)

In [16]:
@tf.function
def extract_embedding(wav_data, label):
  scores, embeddings, spectrogram = yamnet_model(wav_data)
  num_embeddings = tf.shape(embeddings)[0]
  return embeddings, tf.repeat(label, num_embeddings)

In [17]:
model_step1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    tf.keras.layers.Dense(2)
])

model_step2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    tf.keras.layers.Dense(3)
])

In [18]:
model_step1.summary()
model_step2.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 2)                 2050      
                                                                 
Total params: 2,050
Trainable params: 2,050
Non-trainable params: 0
_________________________________________________________________
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 3)                 3075      
                                                                 
Total params: 3,075
Trainable params: 3,075
Non-trainable params: 0
_________________________________________________________________


In [19]:
model_step1.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-5,
              ),
              metrics=['accuracy'],
)

In [20]:
model_step2.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-5,
              ),
              metrics=['accuracy'],
)

In [21]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

In [22]:
train_ds1 = train_ds1.map(load_wav_and_label).map(extract_embedding).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds1 = val_ds1.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds1 = test_ds1.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds2 = train_ds2.map(load_wav_and_label).map(extract_embedding).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds2 = val_ds2.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds2 = test_ds2.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [23]:
history1 = model_step1.fit(train_ds1,
                    epochs=100,
                    validation_data=val_ds1,
                    callbacks=callback
                    )

Epoch 1/100
985/985 [==============================] - 38s 3ms/step - loss: 0.5921 - accuracy: 0.7561 - val_loss: 0.5096 - val_accuracy: 0.8452
Epoch 2/100
985/985 [==============================] - 3s 3ms/step - loss: 0.5217 - accuracy: 0.8237 - val_loss: 0.4498 - val_accuracy: 0.8598
Epoch 3/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4791 - accuracy: 0.8255 - val_loss: 0.4159 - val_accuracy: 0.8585
Epoch 4/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4519 - accuracy: 0.8237 - val_loss: 0.3930 - val_accuracy: 0.8571
Epoch 5/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4331 - accuracy: 0.8256 - val_loss: 0.3766 - val_accuracy: 0.8611
Epoch 6/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4191 - accuracy: 0.8281 - val_loss: 0.3656 - val_accuracy: 0.8624
Epoch 7/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4081 - accuracy: 0.8307 - val_loss: 0.3568 - val_accuracy: 0.867

In [24]:
model_step1.evaluate(test_ds1)

68/68 [==============================] - 4s 45ms/step - loss: 0.2065 - accuracy: 0.8405


[0.2064950317144394, 0.840471088886261]

In [25]:
history2 = model_step2.fit(train_ds2,
                    epochs=100,
                    validation_data=val_ds2,
                    callbacks=callback
                    )

Epoch 1/100
785/785 [==============================] - 32s 4ms/step - loss: 1.1098 - accuracy: 0.4012 - val_loss: 1.1018 - val_accuracy: 0.4202
Epoch 2/100
785/785 [==============================] - 2s 3ms/step - loss: 1.0616 - accuracy: 0.4709 - val_loss: 1.0467 - val_accuracy: 0.4760
Epoch 3/100
785/785 [==============================] - 2s 3ms/step - loss: 1.0260 - accuracy: 0.5052 - val_loss: 1.0077 - val_accuracy: 0.5039
Epoch 4/100
785/785 [==============================] - 2s 3ms/step - loss: 0.9985 - accuracy: 0.5229 - val_loss: 0.9801 - val_accuracy: 0.5302
Epoch 5/100
785/785 [==============================] - 2s 3ms/step - loss: 0.9755 - accuracy: 0.5287 - val_loss: 0.9585 - val_accuracy: 0.5457
Epoch 6/100
785/785 [==============================] - 2s 3ms/step - loss: 0.9555 - accuracy: 0.5328 - val_loss: 0.9391 - val_accuracy: 0.5674
Epoch 7/100
785/785 [==============================] - 2s 3ms/step - loss: 0.9378 - accuracy: 0.5425 - val_loss: 0.9233 - val_accuracy: 0.578

In [26]:
model_step2.evaluate(test_ds2)

58/58 [==============================] - 2s 37ms/step - loss: 0.7211 - accuracy: 0.7447


[0.7210526466369629, 0.7447108626365662]

In [27]:
input_segment = tf.keras.layers.Input(shape=(), dtype=tf.float32, name='audio')
embedding_extraction_layer = hub.KerasLayer(MODEL_HANDLE,
                                            trainable=False, name='yamnet')
_, embeddings, _ = embedding_extraction_layer(input_segment)
step1_output = tf.cast(tf.argmax(tf.math.reduce_mean(model_step1(embeddings), axis=0), axis=0), tf.float32)
step2_output = tf.math.reduce_mean(model_step2(embeddings), axis=0)
final_output = tf.multiply(step1_output, step2_output)

In [28]:
serving_model = tf.keras.Model(input_segment, final_output)

In [29]:
serving_model.save(MODELDIR, include_optimizer=False)

INFO:tensorflow:Assets written to: /Users/egyptdj/github/bruit-classification/twostep_model/assets


INFO:tensorflow:Assets written to: /Users/egyptdj/github/bruit-classification/twostep_model/assets


In [30]:
converter = tf.lite.TFLiteConverter.from_saved_model(MODELDIR)

In [31]:
tflite_model = converter.convert()

In [32]:
with open(os.path.join(MODELDIR, 'bruit_yamnet_twostep.tflite'), 'wb') as f:
  f.write(tflite_model)